# HW03 · Bài 3 · CNN và RNN

So sánh TextCNN với RNN dùng GRU hai chiều. Có thể chọn GPU trong Colab để rút ngắn thời gian huấn luyện.

Phân loại cảm xúc trên dữ liệu IMDB của Stanford. Nhãn 0 là tiêu cực và nhãn 1 là tích cực. Cả năm notebook dùng chung mẫu 200 review lấy phân tầng từ tập test với seed 42 để so sánh.

## Nạp và tiền xử lý dữ liệu

Đọc review có nhãn từ tập IMDB gốc, sắp xếp theo tên tệp để việc lấy mẫu có thể lặp lại và bỏ thẻ HTML xuống dòng trước khi đưa review vào mô hình.

In [ ]:
import re
import shutil
import tarfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_SIZE = 200
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ARCHIVE = Path("data/aclImdb_v1.tar.gz")
LABELS = {"neg": 0, "pos": 1}

if not ARCHIVE.exists():
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    partial = ARCHIVE.with_suffix(".tmp")
    try:
        with urlopen(DATA_URL, timeout=60) as response, partial.open("wb") as file:
            shutil.copyfileobj(response, file)
        partial.replace(ARCHIVE)
    finally:
        partial.unlink(missing_ok=True)


def load_imdb(splits):
    records = {split: [] for split in splits}
    with tarfile.open(ARCHIVE, "r|gz") as archive:
        for member in archive:
            parts = member.name.split("/")
            if not member.isfile() or len(parts) != 4 or parts[0] != "aclImdb":
                continue
            _, split, label, filename = parts
            if split not in records or label not in LABELS or not filename.endswith(".txt"):
                continue
            with archive.extractfile(member) as file:
                text = file.read().decode("utf-8")
            records[split].append((member.name, LABELS[label], text))

    datasets = {}
    for split, rows in records.items():
        if len(rows) != 25_000:
            raise ValueError(f"Tập {split} có {len(rows)} mẫu, dự kiến 25.000.")
        rows.sort(key=lambda row: row[0])
        datasets[split] = (
            [row[2] for row in rows],
            np.array([row[1] for row in rows], dtype=np.int32),
        )
    return datasets


def clean_text(text):
    return re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE).strip()

In [ ]:
data = load_imdb(("train", "test"))
X_train, y_train = data["train"]
X_test, y_test = data["test"]

shared_indices, _ = train_test_split(
    np.arange(len(y_test)), train_size=SAMPLE_SIZE,
    stratify=y_test, random_state=SEED,
)
shared_indices.sort()
print(f"IMDB test: {len(y_test):,} mẫu | Mẫu so sánh: {len(shared_indices)}")

## Độ đo đánh giá

Macro-F1 lấy trung bình F1 của hai nhãn. Accuracy và Macro-F1 được báo cáo trên thang từ 0 đến 1.

In [ ]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred, labels=[0, 1], average="macro", zero_division=0,
        ),
    }


def evaluate(model_name, predictions):
    return [
        {
            "Mô hình": model_name, "Tập đánh giá": "IMDB test", "Số mẫu": len(y_test),
            **scores(y_test, predictions),
        },
        {
            "Mô hình": model_name, "Tập đánh giá": "Mẫu so sánh",
            "Số mẫu": len(shared_indices),
            **scores(y_test[shared_indices], predictions[shared_indices]),
        },
    ]

## Chuỗi token

Tách 20% tập train làm validation theo tỷ lệ nhãn. Từ vựng chỉ học trên phần huấn luyện. Mỗi review giữ tối đa 300 token và được đệm bằng ID `0`.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(SEED)

MAX_TOKENS = 20_000
SEQ_LEN = 300
BATCH_SIZE = 128
EPOCHS = 8

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train,
)
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int", output_sequence_length=SEQ_LEN,
)
fit_texts = tf.constant([clean_text(text) for text in X_fit])
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(fit_texts).batch(BATCH_SIZE))


def token_ids(texts):
    inputs = tf.constant([clean_text(text) for text in texts])
    batches = tf.data.Dataset.from_tensor_slices(inputs).batch(BATCH_SIZE)
    return np.concatenate([vectorizer(batch).numpy() for batch in batches]).astype(np.int32)


fit_ids = token_ids(X_fit)
val_ids = token_ids(X_val)
test_ids = token_ids(X_test)
vocab_size = vectorizer.vocabulary_size()
print(f"Kích thước từ vựng: {vocab_size:,}")

## Hai kiến trúc

CNN dùng tích chập 1D để tìm mẫu cục bộ và global max pooling để tóm tắt chuỗi. RNN dùng GRU hai chiều và bỏ qua vị trí đệm bằng mặt nạ. Cả hai dùng sigmoid để dự đoán xác suất nhãn tích cực.

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,), dtype="int32"),
    layers.Embedding(vocab_size, 64),
    layers.Conv1D(128, kernel_size=5, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="text_cnn")

rnn = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,), dtype="int32"),
    layers.Embedding(vocab_size, 64, mask_zero=True),
    layers.Bidirectional(layers.GRU(64)),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="bidirectional_gru")

## Huấn luyện và đánh giá

Dùng validation loss để dừng sớm và khôi phục trọng số tốt nhất. Trên tập test, dự đoán nhãn tích cực khi xác suất đạt ít nhất 0,5.

In [ ]:
rows = []
for name, model in {"TextCNN": cnn, "RNN (BiGRU)": rnn}.items():
    print(f"Huấn luyện {name}")
    model.summary()
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(
        fit_ids, y_fit,
        validation_data=(val_ids, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True,
        )],
    )
    probabilities = model.predict(test_ids, batch_size=BATCH_SIZE, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(np.int32)
    rows.extend(evaluate(name, predictions))

display(pd.DataFrame(rows).round(4))

## Tài liệu

[IMDB gốc](https://ai.stanford.edu/~amaas/data/sentiment/) · [TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) · [GRU](https://keras.io/api/layers/recurrent_layers/gru/)